# CSC14120 - Final Project - Group 8

## 3.2 Problem understanding

Bài toán nhận hai ảnh RGB của cùng một cảnh, trong đó hai ảnh có vùng chồng lấp khoảng 30-50%, và tạo ra một ảnh panorama duy nhất. Ảnh B được đưa về hệ tọa độ của ảnh A. Nhóm sử dụng sáu stage nối tiếp nhau: preprocessing, phát hiện keypoint, mô tả đặc trưng, matching, ước lượng homography, và warping/blending.

Trong ba stage đầu, mỗi ảnh được xử lý độc lập nên có thể chạy song song ở các phiên bản CUDA sau. Stage 4 cần descriptor của cả hai ảnh để tìm các cặp điểm tương ứng. Stage 5 dùng các match để ước lượng ma trận homography 3x3 ánh xạ tọa độ ảnh B sang ảnh A bằng RANSAC. Stage 6 dùng ma trận này để inverse-warp ảnh B, nội suy màu và hòa trộn vùng overlap với ảnh A.

Với version V0, toàn bộ pipeline chạy tuần tự trên CPU và không dùng OpenMP, std::thread hoặc SIMD viết thủ công. V0 là baseline dùng để kiểm tra tính đúng đắn của panorama và so sánh speedup của các phiên bản GPU. Input/output của chương trình là hai ảnh RGB và một ảnh panorama; thời gian đọc/ghi ảnh được loại khỏi thời gian pipeline.

### Luồng dữ liệu

```text
Image A -> S1 -> S2 -> S3 --\
                              S4 -> S5 -> S6 -> Panorama
Image B -> S1 -> S2 -> S3 --/
```

## 3.3 Chosen methods

| Stage | Phương pháp V0 | Tham số chính | Lý do lựa chọn |
|---|---|---|---|
| S1 Preprocessing | RGB to grayscale + Gaussian blur separable | sigma = 1.4, radius = 4 | Giảm nhiễu trước khi tìm góc. Blur separable giảm số phép tính từ kernel 2D sang hai lượt 1D và là nền tảng thuận lợi cho shared-memory tiling trên GPU. |
| S2 Keypoint detection | Harris response + 5x5 NMS + top-K | Harris k = 0.04, threshold = 1% response lớn nhất, top-K = 3000, border = 16 | Harris cho response liên tục và dễ kiểm tra trên ảnh tổng hợp; NMS loại điểm lân cận trùng nhau; top-K giữ kích thước dữ liệu ổn định cho S3-S4. |
| S3 Description | BRIEF 256-bit trên patch 31x31 | seed = 12345 | Descriptor nhị phân nhỏ, so sánh bằng Hamming distance nhanh và phù hợp với việc chuyển mỗi keypoint thành một CUDA thread. |
| S4 Matching | Brute-force Hamming + Lowe ratio test + cross-check | ratio = 0.8 | Đây là bài toán all-pairs rõ ràng cho baseline; ratio test loại match mơ hồ và cross-check loại các match không đối xứng. |
| S5 Homography | RANSAC + 4-point normalized DLT + inlier refinement | 2000 iterations, threshold = 3 px, seed = 2026 | RANSAC chống outlier từ matching; normalized DLT ổn định hơn khi tọa độ ảnh lớn; refinement dùng lại toàn bộ inlier tốt nhất. |
| S6 Warping and blending | Inverse warp + bilinear interpolation + feather blending | H maps B to A | Inverse mapping bảo đảm mỗi output pixel được lấy mẫu một lần; bilinear giảm răng cưa và feather làm mượt seam trong vùng overlap. |

Tất cả phiên bản phải giữ nguyên các phương pháp và tham số này để timing và chất lượng đầu ra có thể so sánh công bằng. V0 dùng các vòng lặp đơn luồng trên CPU; V1 sẽ ánh xạ trực tiếp các phép tính theo pixel/keypoint/match lên GPU, còn các phiên bản tối ưu sau sẽ thử shared memory, constant memory, kernel fusion, prefix-sum compaction và parallel top-K.